# 13b — MicroEPI Spike Detection (continuous, NO trial separation)

**Goal.** Same cleaning + spike detection as `13_MicroSpikeSorting_Full`, but for an
arbitrary set of NS6 files from **any patient**, with **no photodiode / TSV / trial
timing**. Just: load → per-channel adaptive notch → per-tetrode/per-contact spike
detection → diagnostics → save (`.h5` cache + `FBMdata_*.bin` int16 export).

**Pipeline.**
1. Load one or more NS6 files into a continuous 30 kHz stream (µV). Unused `X` wires dropped.
2. Per-channel adaptive 50-Hz harmonic notch (`lf_notch`, `mode="per_channel"`).
3. Spike detection (`lf_sort.sort_session`): bandpass 300–6000 Hz → intra-tetrode CAR →
   MAD threshold → snippets, with saturation + fragment-boundary guards.
   `DETECTION_UNIT="wire"` = per contact; `"tetrode"` = merged one-event-per-neuron.
4. Per-unit diagnostics: rate over time, waveform overlay, ISI histogram, peak-wire dist.
5. Save `<pid>_spikes.h5` and the whole cleaned micro recording as `FBMdata_DD-MM-YY.bin`.

**Out of scope.** Trials, conditions, PSTH, behaviour. (Use notebook 13 for the trial-locked version.)


In [ ]:
# ---- Imports + config (any patient; no trial separation) ----
import os, sys, re
sys.path.insert(0, "functions")

import numpy as np
import matplotlib.pyplot as plt
import h5py

import lf_micro_io as bb        # NS6 loader (readable; neo.io.BlackrockIO -> uV)
import lf_notch    as nf        # per-channel adaptive mains notch
import lf_sort     as ls        # configurable per-contact/tetrode detection + plots

%load_ext autoreload
%autoreload 2

# -----------------------------------------------------------------------------
# INPUT — pick ONE of two ways to point at your NS6 files:
#   (A) explicit ordered list of file paths   (simplest: "a few ns6 files"), or
#   (B) directory + base filename + ordered suffixes (Blackrock fragment scheme).
# If NS6_FILES is non-empty, (A) is used; otherwise (B).
# -----------------------------------------------------------------------------
patient_id = "MyPatient"        # used only for output filenames

# (A) explicit paths (edit these):
NS6_FILES = [
    # r"\\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\MICROEPI\...\file31.ns6",
    # r"\\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\MICROEPI\...\file32.ns6",
]

# (B) base + suffix scheme (used only if NS6_FILES is empty):
BLACKROCK_DIR = r"\\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\MICROEPI\MicroEPI-G-01\roi_seizures\Seizure1"
BASE_FILENAME = "20241127-161902-12" 
FILE_SUFFIXES = ["15","16"]              # e.g. ["31","32","33"]

out_dir   = os.path.join("outputs", "13b_microSpikeSorting_notrials", patient_id)
out_cache = os.path.join(out_dir, f"{patient_id}_spikes.h5")

# Channel selection / naming
KEEP_PREFIXES            = None             # e.g. ("ADm",) to restrict; None = all micros
DROP_MICRO_PATTERN       = r"^[Xx]\d+[mM]\d+$"
TETRODE_EXCLUDE_PREFIXES = ("X", "x")

# Adaptive notch
DO_NOTCH       = True
NOTCH_BASE_HZ  = 50.0           # 60.0 in the US
NOTCH_FMAX_HZ  = 1000.0       # only notch the mains comb up to here
                              # (higher harmonics are tiny + handled by intra-tetrode CAR; raise if needed)
NOTCH_REPEATS  = 1
NOTCH_Z_THRESH = 3.0

# --- "filter stronger" knobs ---
NOTCH_DF_SAFETY = 2.0     # widen each notch to catch peak skirts (1.0 = exact width)
NOTCH_PASSES    = 3       # re-notch residual peaks up to N passes (auto-stops once clean)
# Optional band-pass of the CLEANED micro signal (None = keep wideband). Removes
# sub-300 Hz mains harmonics + slow drift outright. Spike DETECTION band-passes
# internally regardless; this shapes the saved .h5 / .bin and the PSD QC.
BANDPASS_HZ    = (300.0, 6000.0)
BANDPASS_ORDER = 4

# Spike detection
DETECTION_UNIT    = "wire"      # "wire" = per contact, "tetrode" = merged across 4 wires
SPIKE_LOW_HZ      = 300.0
SPIKE_HIGH_HZ     = 6000.0
K_MAD             = 6.0
SPIKE_POLARITY    = "neg"
REFRACTORY_MS     = 1.5
SNIPPET_PRE_MS    = 0.5
SNIPPET_POST_MS   = 1.5
MIN_AMPLITUDE_UV  = 100.0       # amplitude floor: keep spikes reaching <= -100 uV (None to disable)
MAX_AMPLITUDE_UV  = 300.0
FRAGMENT_GUARD_MS = 100.0

# FBM .bin export
FBM_DATE     = ""               # "" -> auto from first filename (YYYYMMDD) or today; else "DD-MM-YY"
FBM_BIN_GAIN = 1.0              # uV per int16 unit (raise if the .bin reports clipping)

print(f"patient={patient_id}  detection_unit={DETECTION_UNIT}  notch={DO_NOTCH}")
print(f"input: {'explicit NS6_FILES' if NS6_FILES else 'base+suffix scheme'}")


In [ ]:
# ---- Load NS6 -> continuous stream, optional subset, per-channel notch ----
if NS6_FILES:
    session = bb.read_ns6_files(NS6_FILES, drop_micro_pattern=DROP_MICRO_PATTERN, verbose=True)
else:
    session = bb.read_blackrock_session(BLACKROCK_DIR, BASE_FILENAME, FILE_SUFFIXES,
                                        drop_micro_pattern=DROP_MICRO_PATTERN, verbose=True)

fs                    = session["fs"]
signals_micro         = session["signals_micro"]
names_micro           = session["names_micro"]
duration_s            = session["duration_s"]
fragment_boundaries_s = session["fragment_boundaries_s"]

# optional tetrode subset
if KEEP_PREFIXES:
    keep = np.array([any(n.startswith(p) for p in KEEP_PREFIXES) for n in names_micro])
    signals_micro = signals_micro[:, keep]
    names_micro   = [n for n, k in zip(names_micro, keep) if k]
    print(f"[subset] keeping {int(keep.sum())} channels: {names_micro}")
    assert keep.sum() > 0, "No channels matched KEEP_PREFIXES"

print(f"fs={fs} Hz, duration={duration_s/60:.1f} min, micro channels={len(names_micro)}")

# keep a pre-notch copy (same channels) for the PSD QC
signals_micro_pre = signals_micro.copy()

if DO_NOTCH:
    signals_micro, notch_audit = nf.notch_mains_harmonics(
        signals_micro, fs,
        base=NOTCH_BASE_HZ, max_hz=min(NOTCH_FMAX_HZ, 0.5 * fs),
        repeats=NOTCH_REPEATS, peak_z_thresh=NOTCH_Z_THRESH, df_safety=NOTCH_DF_SAFETY, max_passes=NOTCH_PASSES,
        mode="per_channel", return_audit=True, verbose=True,
    )
else:
    print("[notch] skipped (DO_NOTCH=False)")


# ---- Optional band-pass of the cleaned micro signal (spike band) ----
if BANDPASS_HZ:
    signals_micro = ls.bandpass_spike_band(signals_micro, fs,
                                           low=BANDPASS_HZ[0], high=BANDPASS_HZ[1],
                                           order=BANDPASS_ORDER)
    print(f"[bandpass] {BANDPASS_HZ[0]:.0f}-{BANDPASS_HZ[1]:.0f} Hz applied to signals_micro")


In [ ]:
# ---- PSD QC: before vs after notch (+ per-channel residual) ----
from scipy.signal import welch

def _psd(x, fs, sec=60):
    n = min(x.shape[0], int(sec * fs))
    return welch(x[:n], fs=fs, nperseg=int(2 * fs), axis=0)

f, P_pre  = _psd(signals_micro_pre, fs)
_, P_post = _psd(signals_micro,     fs)

fig, ax = plt.subplots(figsize=(14, 4))
ax.loglog(f, P_pre.mean(1),  label="pre")
ax.loglog(f, P_post.mean(1), label="post")
for h in np.arange(50, fs / 2, 50):
    ax.axvline(h, c="r", lw=.3, alpha=.3)
ax.set(xlabel="Hz", ylabel=r"PSD [$\mu V^2$/Hz]"); ax.legend(); ax.grid(alpha=.3)
plt.show()

# per-channel residual after notch (should be ~0 dB where a harmonic was present)
f0s, resid = nf.harmonic_residual_db(signals_micro, fs, base=NOTCH_BASE_HZ,
                                     max_hz=min(800.0, 0.5 * fs))

print("Per-channel post-notch residual (dB) at harmonics [worst / mean channel]:")
for k, f0 in enumerate(f0s):
    col = resid[:, k]
    if np.any(np.isfinite(col)):
        print(f"  {f0:6.0f} Hz   worst {np.nanmax(col):+6.1f}   mean {np.nanmean(col):+6.1f}")


In [ ]:
# ---- Spike detection (continuous; no trial separation) ----
spikes_per_tetrode = ls.sort_session(
    signals_micro, names_micro, fs,
    detection_unit=DETECTION_UNIT, exclude_prefixes=TETRODE_EXCLUDE_PREFIXES,
    spike_low_hz=SPIKE_LOW_HZ, spike_high_hz=SPIKE_HIGH_HZ,
    k_mad=K_MAD, polarity=SPIKE_POLARITY, refractory_ms=REFRACTORY_MS,
    snippet_pre_ms=SNIPPET_PRE_MS, snippet_post_ms=SNIPPET_POST_MS,
    min_amplitude_uv=MIN_AMPLITUDE_UV, max_amplitude_uv=MAX_AMPLITUDE_UV,
    fragment_boundaries_s=fragment_boundaries_s, fragment_guard_ms=FRAGMENT_GUARD_MS,
    verbose=True,
)

print(f"\n{'='*64}")
print(f"DETECTION_UNIT={DETECTION_UNIT!r}  ->  {len(spikes_per_tetrode)} unit(s)")
print(f"{'='*64}")
for lbl, d in spikes_per_tetrode.items():
    rate = d['spike_times_s'].size / duration_s if duration_s else 0.0
    print(f"  {lbl:>18s}: {d['n_final']:>6d} spikes  ({rate:4.1f} Hz, raw {d['n_raw']})")


In [ ]:
# ---- Per-unit diagnostics: rate / waveforms / ISI / peak-wire (no trials) ----
n_u = len(spikes_per_tetrode)
fig, axes = plt.subplots(max(1, n_u), 4, figsize=(16, 3.0 * max(1, n_u)),
                         gridspec_kw=dict(width_ratios=[2.5, 1.3, 1.5, 1.0]))
if n_u <= 1:
    axes = np.atleast_2d(axes)

for ti, (lbl, d) in enumerate(spikes_per_tetrode.items()):
    st = np.asarray(d['spike_times_s']); pc = np.asarray(d['peak_channel'], int)
    wn = d['wire_names']

    ls.plot_spike_rate_over_time(st, duration_s, bin_s=10.0, ax=axes[ti, 0],
                                 title=f"{lbl}  n={st.size}  {st.size/max(duration_s,1e-9):.1f} Hz")
    ls.plot_snippet_overlay(d['snippets'], fs, peak_channel=pc, max_show=300,
                            ax=axes[ti, 1], title=f"{lbl} waveforms")
    if st.size >= 2:
        isi = np.diff(np.sort(st)) * 1000.0
        axes[ti, 2].hist(isi, bins=np.logspace(-1, 4, 60), color="steelblue", alpha=.8)
        axes[ti, 2].axvline(2.0, color="orange", ls="--", lw=.8)
        rv = float((isi < 2.0).sum()) / isi.size * 100.0
        axes[ti, 2].set_xscale("log"); axes[ti, 2].set_xlabel("ISI (ms)")
        axes[ti, 2].set_title(f"ISI  <2ms: {rv:.1f}%")
    else:
        axes[ti, 2].set_title("ISI (n<2)")
    counts = np.array([(pc == ci).sum() for ci in range(len(wn))])
    axes[ti, 3].bar(range(len(wn)), counts)
    axes[ti, 3].set_xticks(range(len(wn)))
    axes[ti, 3].set_xticklabels(wn, rotation=45, fontsize=8)
    axes[ti, 3].set_ylabel("# spikes"); axes[ti, 3].set_title("peak-wire")

plt.tight_layout(); plt.show()


In [ ]:
# ---- Dot raster of spike activity (all contacts) ----
fig, ax = plt.subplots(figsize=(16, max(2, 0.35 * len(spikes_per_tetrode))))
ls.plot_activity_raster(spikes_per_tetrode, duration_s=duration_s, ax=ax,
                        title=f"{patient_id} - spike raster ({DETECTION_UNIT})")
plt.tight_layout(); plt.show()


In [ ]:
# ---- Save .h5 cache (session + spikes; no trial info) ----
os.makedirs(out_dir, exist_ok=True)
with h5py.File(out_cache, "w") as fcache:
    g = fcache.create_group("session")
    g.create_dataset("signals_micro", data=signals_micro, compression="gzip", compression_opts=4)
    g.create_dataset("fragment_boundaries_s", data=fragment_boundaries_s)
    g.attrs["fs"] = fs
    g.attrs["duration_s"] = duration_s
    g.attrs["names_micro"] = np.array(names_micro, dtype="S")
    g.attrs["notch_applied"] = DO_NOTCH

    gs = fcache.create_group("spikes")
    for lbl, d in spikes_per_tetrode.items():
        gt = gs.create_group(lbl)
        gt.create_dataset("spike_times_s", data=d["spike_times_s"])
        gt.create_dataset("snippets", data=d["snippets"], compression="gzip", compression_opts=4)
        gt.create_dataset("peak_channel", data=d["peak_channel"])
        gt.create_dataset("thresholds", data=d["thresholds"])
        gt.attrs["wire_names"] = np.array(d["wire_names"], dtype="S")

    fcache.attrs["patient_id"] = patient_id
    fcache.attrs["detection_unit"] = DETECTION_UNIT
    fcache.attrs["k_mad"] = K_MAD
    fcache.attrs["polarity"] = SPIKE_POLARITY

print(f"Saved {out_cache}  ({os.path.getsize(out_cache)/1e9:.2f} GB)")


In [ ]:
# ---- Export whole cleaned micro recording as int16 .bin (FBMdata) ----
# int16, little-endian, C-order (n_samples, n_channels); JSON sidecar describes it.
import json, datetime as _dt

def _fbm_date():
    if FBM_DATE:
        return FBM_DATE
    base = os.path.basename(str(NS6_FILES[0] if NS6_FILES else BASE_FILENAME))
    m = re.search(r"(\d{4})(\d{2})(\d{2})", base)
    if m:
        y, mo, d = m.groups(); return f"{d}-{mo}-{y[2:]}"
    return _dt.date.today().strftime("%d-%m-%y")

date_str  = _fbm_date()
os.makedirs(out_dir, exist_ok=True)
bin_path  = os.path.join(out_dir, f"FBMdata_{date_str}.bin")
meta_path = os.path.join(out_dir, f"FBMdata_{date_str}.json")

_scaled = np.asarray(signals_micro, dtype=np.float64) / FBM_BIN_GAIN
_n_clip = int(np.sum((_scaled < -32768) | (_scaled > 32767)))
_int16  = np.clip(np.round(_scaled), -32768, 32767).astype("<i2")
_int16.tofile(bin_path)

json.dump(dict(
    file=os.path.basename(bin_path), dtype="int16", byte_order="little", order="C",
    shape=list(_int16.shape), axes=["n_samples", "n_channels"], fs=float(fs),
    n_samples=int(_int16.shape[0]), n_channels=int(_int16.shape[1]),
    channel_names=list(names_micro), units="microvolts",
    gain_uV_per_unit=float(FBM_BIN_GAIN), notch_applied=bool(DO_NOTCH),
    n_samples_clipped=_n_clip,
), open(meta_path, "w"), indent=2)

print(f"[FBM .bin] {bin_path}")
print(f"           {_int16.shape[0]} samples x {_int16.shape[1]} ch int16 "
      f"({os.path.getsize(bin_path)/1e9:.2f} GB), gain={FBM_BIN_GAIN} uV/unit, clipped={_n_clip}")
print(f"[FBM .bin] sidecar: {meta_path}")
print("           reload: np.fromfile(path, dtype='<i2').reshape(n_samples, n_channels)")
